In [ ]:
from __future__ import annotations

from app.utils.helpers import generate_id, utc_now, record_activity


class RecommendationService:
    """
    Generates actionable learning recommendations from
    persistent learner state.

    Recommendations are generated from:
        - mastery
        - recent assessments
        - recent activity
        - project goals
        - available materials

    Generated recommendations are persisted in MongoDB.
    """

    def __init__(self, database):
        self.database = database

    # ========================================================
    # GENERATE
    # ========================================================

    def generate_for_project(
        self,
        user_id: str,
        project_id: str,
        goals: list[str] | None = None,
    ):

        from app.services.ai_service import AIService
        from app.ai.recommender import Recommender

        goals = goals or []

        # ----------------------------------------------------
        # MASTERY
        # ----------------------------------------------------

        mastery = list(
            self.database.collection(
                "mastery"
            ).find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                },
            ).sort(
                "score",
                1,
            ).limit(30)
        )

        # ----------------------------------------------------
        # RECENT ACTIVITY
        # ----------------------------------------------------

        recent_activity = list(
            self.database.collection(
                "activities"
            ).find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                },
            ).sort(
                "created_at",
                -1,
            ).limit(30)
        )

        # ----------------------------------------------------
        # RECENT ASSESSMENTS
        # ----------------------------------------------------

        recent_performance = list(
            self.database.collection(
                "assessments"
            ).find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                },
            ).sort(
                "created_at",
                -1,
            ).limit(10)
        )

        # ----------------------------------------------------
        # AVAILABLE MATERIAL
        # ----------------------------------------------------
        #
        # We deliberately send metadata rather than entire PDFs
        # to avoid unnecessary token usage.
        # ----------------------------------------------------

        materials = list(
            self.database.collection(
                "materials"
            ).find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                },
                {
                    "_id": 0,
                    "id": 1,
                    "file_name": 1,
                    "title": 1,
                    "processing_status": 1,
                    "page_count": 1,
                    "chunk_count": 1,
                },
            ).limit(30)
        )

        available_material = []

        for material in materials:

            available_material.append(
                {
                    "material_id": material.get(
                        "id"
                    ),
                    "file_name": material.get(
                        "file_name"
                    ),
                    "title": material.get(
                        "title"
                    ),
                    "processing_status": material.get(
                        "processing_status"
                    ),
                    "page_count": material.get(
                        "page_count"
                    ),
                    "chunk_count": material.get(
                        "chunk_count"
                    ),
                }
            )

        # ----------------------------------------------------
        # AI
        # ----------------------------------------------------

        ai = AIService(
            database=self.database
        )

        recommender = Recommender(
            ai_service=ai
        )

        try:
            print(
                "[RecommendationService] "
                f"project_id={project_id} "
                f"mastery_count={len(mastery)} "
                f"activity_count={len(recent_activity)} "
                f"assessment_count={len(recent_performance)} "
                f"material_count={len(materials)}"
            )
            result = recommender.generate(
                mastery=mastery,
                recent_performance=recent_performance,
                goals=goals,
                recent_activity=recent_activity,
                user_id=user_id,
                project_id=project_id,
            )

        except Exception as exc:
            print(
                "[RecommendationService] generation failed: "
                f"{type(exc).__name__}: {exc}"
            )

            raise RuntimeError(
                "Failed to generate recommendations."
            ) from exc

        # ----------------------------------------------------
        # PERSIST WITH DEDUPLICATION
        # ----------------------------------------------------

        collection = self.database.collection(
            "recommendations"
        )

        documents = []

        for recommendation in result.recommendations:

            title = str(
                recommendation.title
            ).strip()

            reason = str(
                recommendation.reason
            ).strip()

            action = str(
                recommendation.action
            ).strip()

            if not title or not action:
                continue

            # --------------------------------------------------------
            # NORMALIZE ENUM-LIKE VALUES
            # --------------------------------------------------------

            priority = str(
                recommendation.priority or "medium"
            ).strip().lower()

            if priority not in {
                "low",
                "medium",
                "high",
            }:
                priority = "medium"

            concept_ids = (
                recommendation.concept_ids
                if isinstance(
                    recommendation.concept_ids,
                    list,
                )
                else []
            )

            source_types = (
                recommendation.source_types
                if isinstance(
                    recommendation.source_types,
                    list,
                )
                else []
            )

            concept_ids = [
                str(value).strip()
                for value in concept_ids
                if value is not None
                and str(value).strip()
            ]

            source_types = [
                str(value).strip()
                for value in source_types
                if value is not None
                and str(value).strip()
            ]

            # --------------------------------------------------------
            # DEDUPLICATION
            # --------------------------------------------------------

            existing = collection.find_one(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                    "title": title,
                    "action": action,
                    "status": {
                        "$in": [
                            "pending",
                            "shown",
                        ]
                    },
                },
                {
                    "_id": 0,
                },
            )

            if existing is not None:
                documents.append(existing)
                continue

            # --------------------------------------------------------
            # DOCUMENT
            # --------------------------------------------------------

            now = utc_now()

            document = {
                "id": generate_id(),
                "user_id": user_id,
                "project_id": project_id,
                "title": title,
                "reason": reason,
                "action": action,
                "priority": priority,
                "concept_ids": concept_ids,
                "source_types": source_types,
                "status": "pending",
                "created_at": now,
                "updated_at": now,
            }

            try:
                collection.insert_one(
                    document
                )

                record_activity(
                    self.database,
                    user_id=user_id,
                    project_id=project_id,
                    event_type="RECOMMENDATION_CREATED",
                    description=f"Created recommendation: {title}",
                    entity_type="recommendation",
                    entity_id=document["id"],
                    metadata={
                        "priority": priority,
                        "concept_ids": concept_ids,
                    },
                )

            except Exception as exc:
                print(
                    "[RecommendationService] "
                    "MongoDB persistence failed: "
                    f"{type(exc).__name__}: {exc}"
                )

                raise RuntimeError(
                    "Failed to save generated recommendations."
                ) from exc

            documents.append(
                document
            )

        return documents
    # ========================================================
    # PENDING
    # ========================================================

    def get_pending(
        self,
        user_id: str,
        project_id: str,
        limit: int = 10,
    ) -> list[dict]:

        limit = max(
            1,
            min(
                int(limit),
                50,
            ),
        )

        return list(
            self.database.collection(
                "recommendations"
            ).find(
                {
                    "user_id": user_id,
                    "project_id": project_id,
                    "status": "pending",
                },
                {
                    "_id": 0,
                },
            ).sort(
                "created_at",
                -1,
            ).limit(limit)
        )
